# 🎯 Marketing Analytics Platform — Complete
### Modules 01–13 | Predictive Campaign Optimization · Causal Uplift · Agentic AI
---
**Author:** Latha Iyer · M.S. Business Analytics, University of Louisville  
**GitHub:** `github.com/lathaiyer/marketing-analytics-platform`  
**Stack:** Python · pandas · scikit-learn · XGBoost · Anthropic Claude API · statsmodels · Streamlit


---

## 📋 Table of Contents

| Module | Description |
|--------|-------------|
| [00 — Data Source](#module-00) | Auto-load real dataset · synthetic fallback |
| [01 — Setup & Data](#module-01) | Install packages · load data |
| [01b — Data Cleansing](#module-01b) | **Missing values · Outliers · Dirty categoricals · Encoding** | | Install packages · real/synthetic dataset loader |
| [02 — Feature Engineering](#module-02) | 22 derived features · RFM signals · MinMax scaling |
| [03 — EDA](#module-03) | Response by segment · channel distribution · scatter plots |
| [04 — Correlation Analysis](#module-04) | Heatmap · top predictors of response |
| [05 — Random Forest](#module-05) | Classifier · feature importance · confusion matrix |
| [06 — XGBoost](#module-06) | Boosted model · ROC comparison · model metrics |
| [07 — Recommender System](#module-07) | SVD · content-based · ALS hybrid |
| [08 — Budget Simulator](#module-08) | Channel ROI · $50K allocation engine |
| [09 — Model Summary](#module-09) | Final dashboard · business recommendations |
| [10 — LLM Narrator](#module-10) | **🤖** Claude API auto-narrates every chart |
| [11 — A/B Testing Agent](#module-11) | **🤖** Power analysis · sequential monitoring · launch decision |
| [12 — Causal Uplift](#module-12) | **🤖** T-Learner · Persuadable segments · Qini curve |
| [13 — Multi-Agent Pipeline](#module-13) | **🤖** DataAgent → ModelAgent → ReportAgent → CMO report |


---
<a name="module-00"></a>
## 📡 Module 00 — Data Source & Data Dictionary

### Official Data Dictionary (from iFood zip)

| Feature | Description |
|---------|-------------|
| **Target** | |
| `Response` | 1 if customer accepted the offer in the **last** campaign, 0 otherwise |
| **Campaign History** | |
| `AcceptedCmp1` | 1 if customer accepted the offer in the 1st campaign, 0 otherwise |
| `AcceptedCmp2` | 1 if customer accepted the offer in the 2nd campaign, 0 otherwise |
| `AcceptedCmp3` | 1 if customer accepted the offer in the 3rd campaign, 0 otherwise |
| `AcceptedCmp4` | 1 if customer accepted the offer in the 4th campaign, 0 otherwise |
| `AcceptedCmp5` | 1 if customer accepted the offer in the 5th campaign, 0 otherwise |
| **Customer Profile** | |
| `Education` | Customer's level of education |
| `Marital` | Customer's marital status |
| `Income` | Customer's yearly household income |
| `Kidhome` | Number of small children in customer's household |
| `Teenhome` | Number of teenagers in customer's household |
| `DtCustomer` | Date of customer's enrollment with the company |
| `Recency` | Number of days since the last purchase |
| `Complain` | 1 if customer complained in the last 2 years |
| **Spend (last 2 years)** | |
| `MntWines` | Amount spent on wines |
| `MntMeatProducts` | Amount spent on meat products |
| `MntFishProducts` | Amount spent on fish products |
| `MntFruits` | Amount spent on fruits |
| `MntSweetProducts` | Amount spent on sweet products |
| `MntGoldProds` | Amount spent on *gold* products |
| **Purchase Channels** | |
| `NumDealsPurchases` | Number of purchases made with discount |
| `NumCatalogPurchases` | Number of purchases made using catalogue |
| `NumStorePurchases` | Number of purchases made directly in stores |
| `NumWebPurchases` | Number of purchases made through company's web site |
| `NumWebVisitsMonth` | Number of visits to company's web site in the last month |

> **Note on ifood_df column names:**  
> The raw dictionary uses `DtCustomer` and `Marital` — the `ifood_df.csv` file uses  
> `Dt_Customer` / `Marital_Status` (with underscores). Our loader handles both automatically.

---

### Data Source

| | Dataset | Rows | Format | Source |
|--|---------|------|--------|--------|
| ✅ **Real** (preferred) | ifood_df.csv | 2,240 | Pre-processed (Age, Customer_Days, one-hot encoded) | iFood data challenge zip |
| 🔄 **Synthetic** (fallback) | Generated | 2,240 | Mirrors real columns exactly | Built-in |

> **To use `ifood_df.csv`:**  
> 1. Upload the file in Colab (📁 sidebar → Upload)  
> 2. Set `UPLOADED_FILE = '/content/ifood_df.csv'` in the cell below  
> 3. Run All


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  MODULE 00 — SMART DATA LOADER
#
#  Supports THREE formats automatically:
#    1. ifood_df.csv        — pre-engineered (Age, Customer_Days, one-hot encoded)
#    2. marketing_data.csv  — raw format (Year_Birth, Dt_Customer, text columns)
#    3. Synthetic fallback  — built-in if no file available
#
#  ➡  TO USE YOUR FILE: set UPLOADED_FILE to its Colab path, e.g.
#       UPLOADED_FILE = '/content/ifood_df.csv'
# ════════════════════════════════════════════════════════════════════════════
import pandas as pd, numpy as np

UPLOADED_FILE = None   # ← set this to your file path in Colab

REAL_URLS = [
    "https://raw.githubusercontent.com/nailson/ifood-data-business-analyst-test/master/ifood_df.csv",
    "https://raw.githubusercontent.com/dsrscientist/dataset1/master/marketing_campaign.csv",
]

# ── Detect which format the file is ──────────────────────────────────────────
def detect_format(df):
    if 'Age' in df.columns and 'marital_Divorced' in df.columns:
        return 'ifood'
    elif 'Year_Birth' in df.columns and 'Marital_Status' in df.columns:
        return 'raw'
    return 'unknown'

# ── Convert ifood_df format → our pipeline format ────────────────────────────
def convert_ifood(df):
    """
    ifood_df already has Age, Customer_Days, and one-hot encoded
    Marital_Status / Education. Reconstruct what our pipeline expects.
    """
    out = df.copy()

    # Drop extra pre-computed columns (we'll recompute them)
    drop_cols = ['Z_CostContact','Z_Revenue','MntTotal','MntRegularProds',
                 'AcceptedCmpOverall']
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])

    # Reconstruct Year_Birth from Age
    if 'Year_Birth' not in out.columns:
        out['Year_Birth'] = 2024 - out['Age'].astype(int)

    # Reconstruct Marital_Status from one-hot columns
    if 'Marital_Status' not in out.columns:
        marital_cols = {
            'marital_Divorced':'Divorced', 'marital_Married':'Married',
            'marital_Single':'Single',   'marital_Together':'Together',
            'marital_Widow':'Widow'
        }
        available = {k:v for k,v in marital_cols.items() if k in out.columns}
        if available:
            def get_marital(row):
                for col, label in available.items():
                    if row.get(col, 0) == 1:
                        return label
                return 'Single'
            out['Marital_Status'] = out.apply(get_marital, axis=1)

    # Reconstruct Education from one-hot columns
    if 'Education' not in out.columns:
        edu_cols = {
            'education_Basic':'Basic', 'education_2n Cycle':'2n Cycle',
            'education_Graduation':'Graduation', 'education_Master':'Master',
            'education_PhD':'PhD'
        }
        available = {k:v for k,v in edu_cols.items() if k in out.columns}
        if available:
            def get_edu(row):
                for col, label in available.items():
                    if row.get(col, 0) == 1:
                        return label
                return 'Graduation'
            out['Education'] = out.apply(get_edu, axis=1)

    # Reconstruct Dt_Customer from Customer_Days
    if 'Dt_Customer' not in out.columns and 'Customer_Days' in out.columns:
        ref = pd.Timestamp('2024-01-01')
        out['Dt_Customer'] = out['Customer_Days'].apply(
            lambda d: (ref - pd.Timedelta(days=int(d))).strftime('%Y-%m-%d')
        )

    # Fix Income: strip $ signs and commas if string
    if out['Income'].dtype == object:
        out['Income'] = out['Income'].str.replace(r'[$,]','',regex=True).astype(float)

    # Fill missing income
    out['Income'] = pd.to_numeric(out['Income'], errors='coerce')
    out['Income'] = out['Income'].fillna(out['Income'].median())

    # Remove extreme outlier birth years (e.g. 1893)
    out = out[out['Year_Birth'] >= 1900].copy()

    return out.reset_index(drop=True)

# ── Convert raw marketing_data format ────────────────────────────────────────
def convert_raw(df):
    out = df.copy()
    out = out.drop(columns=[c for c in ['Z_CostContact','Z_Revenue'] if c in out.columns])
    out['Income'] = pd.to_numeric(
        out['Income'].astype(str).str.replace(r'[$,]','',regex=True), errors='coerce')
    out['Income'] = out['Income'].fillna(out['Income'].median())
    out = out[out['Year_Birth'] >= 1900].copy()
    return out.reset_index(drop=True)

# ── Synthetic fallback ────────────────────────────────────────────────────────
def make_synthetic(n=2240):
    np.random.seed(42)
    income    = np.random.normal(52000,21000,n).clip(10000,120000)
    age       = np.random.randint(18,80,n)
    recency   = np.random.randint(0,100,n)
    education = np.random.choice(['Basic','2n Cycle','Graduation','Master','PhD'],
                                  n, p=[0.05,0.10,0.50,0.20,0.15])
    marital   = np.random.choice(['Single','Together','Married','Divorced','Widow'],
                                  n, p=[0.22,0.26,0.38,0.10,0.04])
    kidhome   = np.random.choice([0,1,2],n,p=[0.57,0.35,0.08])
    teenhome  = np.random.choice([0,1,2],n,p=[0.55,0.36,0.09])
    mnt_wines = (income*np.random.uniform(0.001,0.018,n)).clip(0,1500).astype(int)
    mnt_fruit = np.random.randint(0,200,n)
    mnt_meat  = (income*np.random.uniform(0.001,0.014,n)).clip(0,1200).astype(int)
    mnt_fish  = np.random.randint(0,250,n)
    mnt_sweet = np.random.randint(0,250,n)
    mnt_gold  = np.random.randint(0,400,n)
    p_respond = (0.15+0.25*(income/120000)-0.15*(recency/100)
                 +0.20*(mnt_wines/1500)+np.random.normal(0,0.05,n)).clip(0.02,0.95)
    response  = (np.random.rand(n)<p_respond).astype(int)
    cmp       = {f'AcceptedCmp{i}':(np.random.rand(n)<0.06).astype(int) for i in range(1,6)}
    dt        = pd.date_range('2012-01-01',periods=n,freq='1D')[:n]
    dt        = np.random.choice(dt,n,replace=False)
    return pd.DataFrame(dict(
        ID=range(1,n+1), Year_Birth=2024-age, Education=education,
        Marital_Status=marital, Income=income.astype(int),
        Kidhome=kidhome, Teenhome=teenhome, Dt_Customer=dt, Recency=recency,
        MntWines=mnt_wines, MntFruits=mnt_fruit, MntMeatProducts=mnt_meat,
        MntFishProducts=mnt_fish, MntSweetProducts=mnt_sweet, MntGoldProds=mnt_gold,
        NumDealsPurchases=np.random.randint(0,15,n),
        NumWebPurchases=np.random.randint(0,14,n),
        NumCatalogPurchases=np.random.randint(0,14,n),
        NumStorePurchases=np.random.randint(0,14,n),
        NumWebVisitsMonth=np.random.randint(1,20,n),
        **cmp, Response=response,
        Complain=(np.random.rand(n)<0.01).astype(int)
    ))

# ══════════════════════════════════════════════════════════════════════════════
#  LOAD — try in order: upload → URL → synthetic
# ══════════════════════════════════════════════════════════════════════════════
DATA_SOURCE = None
df_raw = None

# 1. Uploaded file
if UPLOADED_FILE:
    try:
        raw = pd.read_csv(UPLOADED_FILE, sep=None, engine='python')
        fmt = detect_format(raw)
        if fmt == 'ifood':
            df_raw = convert_ifood(raw)
            DATA_SOURCE = f'📂 ifood_df format — {UPLOADED_FILE}'
        elif fmt == 'raw':
            df_raw = convert_raw(raw)
            DATA_SOURCE = f'📂 Raw marketing format — {UPLOADED_FILE}'
        else:
            print(f"⚠️  Unrecognised format. Columns: {list(raw.columns[:8])}")
    except Exception as e:
        print(f"⚠️  Upload error: {e}")

# 2. URL download
if df_raw is None:
    for url in REAL_URLS:
        try:
            raw = pd.read_csv(url, sep=None, engine='python')
            fmt = detect_format(raw)
            if fmt in ('ifood','raw') and 'Response' in raw.columns and len(raw) > 1000:
                df_raw = convert_ifood(raw) if fmt=='ifood' else convert_raw(raw)
                DATA_SOURCE = f'🌐 {fmt.upper()} format — {url.split("/")[-1]}'
                break
        except Exception:
            continue

# 3. Synthetic fallback
if df_raw is None:
    df_raw = make_synthetic()
    DATA_SOURCE = '🔄 Synthetic dataset (upload ifood_df.csv to use real data)'

N = len(df_raw)

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 62)
print(f"  DATA SOURCE : {DATA_SOURCE}")
print("=" * 62)
print(f"  Rows        : {N:,}")
print(f"  Columns     : {df_raw.shape[1]}")
print(f"  Response +  : {df_raw['Response'].sum():,}  ({df_raw['Response'].mean():.1%})")
print(f"  Response -  : {(df_raw['Response']==0).sum():,}  ({(df_raw['Response']==0).mean():.1%})")
print(f"  Income range: ${df_raw['Income'].min():,.0f} – ${df_raw['Income'].max():,.0f}")
print(f"  Missing vals: {df_raw.isnull().sum().sum()}")
print("=" * 62)
print()
print("✅ To use your ifood_df.csv:")
print("   1. Upload file in Colab (📁 sidebar → Upload)")
print("   2. Change line above to:  UPLOADED_FILE = '/content/ifood_df.csv'")
print("   3. Re-run this cell")
df_raw.head(3)


---
<a name="module-01"></a>
## 📦 Module 01 — Setup & Data Loading


In [ ]:
# Install required packages
!pip install -q xgboost scikit-learn imbalanced-learn anthropic scipy statsmodels

import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd

# Viz
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# ML
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report,
                              confusion_matrix, precision_score, recall_score, f1_score)
from sklearn.decomposition import TruncatedSVD
from xgboost import XGBClassifier

# Agentic AI
import json, textwrap
import anthropic
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# Style
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#0A9396', '#EE9B00', '#0D3B6E', '#AE2012', '#94D2BD', '#005F73']
sns.set_palette(PALETTE)

print("✅ All packages loaded successfully")


---
<a name="module-01b"></a>
## 🧹 Module 01b — Data Cleansing, Outlier Treatment & Encoding

> **Why this matters for Paramark / ML interviews:**  
> Garbage in → garbage out. Every ML metric downstream depends on the quality of this step.

**Pipeline:**
1. Missing value audit
2. Outlier detection (IQR + visual)
3. Outlier treatment (capping)
4. Dirty categorical values (Marital_Status)
5. Encoding strategy decision
6. Clean data summary


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — MISSING VALUE AUDIT
# ══════════════════════════════════════════════════════════════════════════════
df = df_raw.copy()

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("=" * 50)
print("  MISSING VALUE REPORT")
print("=" * 50)
if len(missing) == 0:
    print("  ✅ No missing values found")
else:
    for col, cnt in missing.items():
        pct = cnt / len(df) * 100
        bar = '█' * int(pct * 2)
        print(f"  {col:25s}: {cnt:4d} rows  ({pct:.1f}%)  {bar}")
print("=" * 50)
print(f"  Total rows    : {len(df):,}")
print(f"  Total missing : {df.isnull().sum().sum()}")

# Visualise missing pattern
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Heatmap of nulls
null_matrix = df.isnull()
cols_with_nulls = null_matrix.columns[null_matrix.any()].tolist()
if cols_with_nulls:
    sns.heatmap(null_matrix[cols_with_nulls].T, cbar=False,
                ax=axes[0], cmap='viridis', yticklabels=True)
    axes[0].set_title('Missing Value Pattern', fontweight='bold')
    axes[0].set_xlabel('Row index')
else:
    axes[0].text(0.5, 0.5, '✅ No Missing Values', ha='center', va='center',
                 fontsize=14, transform=axes[0].transAxes)
    axes[0].set_title('Missing Value Pattern', fontweight='bold')
    axes[0].axis('off')

# Distribution of Income (to justify median imputation)
df['Income'].dropna().hist(ax=axes[1], bins=40, color=PALETTE[0],
                            edgecolor='white', linewidth=0.8)
axes[1].axvline(df['Income'].median(), color=PALETTE[1], lw=2.5,
                linestyle='--', label=f"Median: ${df['Income'].median():,.0f}")
axes[1].axvline(df['Income'].mean(), color=PALETTE[3], lw=2.5,
                linestyle='--', label=f"Mean: ${df['Income'].mean():,.0f}")
axes[1].set_title('Income Distribution (skewed → use median imputation)', fontweight='bold')
axes[1].set_xlabel('Income ($)'); axes[1].legend()
sns.despine(ax=axes[1])

plt.tight_layout(); plt.show()

# ── Fix: median imputation for Income ────────────────────────────────────────
income_median = df['Income'].median()
n_imputed = df['Income'].isnull().sum()
df['Income'] = df['Income'].fillna(income_median)
print(f"\n✅ Imputed {n_imputed} missing Income values with median ${income_median:,.0f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — OUTLIER DETECTION (IQR method)
# ══════════════════════════════════════════════════════════════════════════════
numeric_cols = ['Income', 'Age', 'MntWines', 'MntMeatProducts',
                'MntFishProducts', 'MntFruits', 'MntSweetProducts',
                'MntGoldProds', 'NumWebVisitsMonth']

# Compute Age first
df['Age'] = 2024 - df['Year_Birth']

print("=" * 65)
print(f"  {'Column':22s}  {'Min':>8}  {'Max':>10}  {'IQR Low':>10}  {'IQR High':>10}  {'Outliers':>8}")
print("=" * 65)
outlier_report = {}
for col in numeric_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo  = Q1 - 1.5 * IQR
    hi  = Q3 + 1.5 * IQR
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    outlier_report[col] = {'Q1':Q1,'Q3':Q3,'IQR_lo':lo,'IQR_hi':hi,'n_outliers':n_out}
    flag = ' ⚠️ ' if n_out > 0 else '   '
    print(f"  {flag}{col:20s}  {df[col].min():>8.0f}  {df[col].max():>10.0f}"
          f"  {lo:>10.0f}  {hi:>10.0f}  {n_out:>8}")
print("=" * 65)

# Visualise — boxplots BEFORE treatment
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Outlier Detection — Boxplots BEFORE Treatment', fontsize=14, fontweight='bold')
axes = axes.flatten()
for ax, col in zip(axes, numeric_cols[:8]):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor=PALETTE[0], alpha=0.7),
               medianprops=dict(color=PALETTE[1], lw=2.5),
               flierprops=dict(marker='o', markerfacecolor=PALETTE[3],
                               markersize=3, alpha=0.5))
    ax.set_title(col, fontweight='bold', fontsize=9)
    sns.despine(ax=ax)
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3 — OUTLIER TREATMENT
# ══════════════════════════════════════════════════════════════════════════════
#
#  Strategy (deliberate choices):
#  ┌──────────────────────┬────────────────────────────────────────────────────┐
#  │ Column               │ Treatment                                          │
#  ├──────────────────────┼────────────────────────────────────────────────────┤
#  │ Age > 90             │ DROP — likely data entry errors (born ~1893-1900)  │
#  │ Income > 99th pctile │ CAP — extreme values are real but distort scaling  │
#  │ Spend columns        │ CAP at 99th percentile — heavy spenders are real   │
#  │ NumWebVisitsMonth    │ CAP at 99th percentile                             │
#  └──────────────────────┴────────────────────────────────────────────────────┘

df_clean = df.copy()
rows_before = len(df_clean)

# 1. Drop extreme age outliers (Age > 90)
age_outliers = (df_clean['Age'] > 90).sum()
df_clean = df_clean[df_clean['Age'] <= 90].copy()
print(f"🗑️  Dropped {age_outliers} rows with Age > 90 (birth year < 1934)")

# 2. Cap Income at 99th percentile
p99_income = df_clean['Income'].quantile(0.99)
income_capped = (df_clean['Income'] > p99_income).sum()
df_clean['Income'] = df_clean['Income'].clip(upper=p99_income)
print(f"📌 Capped {income_capped} Income values above ${p99_income:,.0f} (99th pct)")

# 3. Cap spend columns at 99th percentile
spend_cols = ['MntWines','MntMeatProducts','MntFishProducts',
              'MntFruits','MntSweetProducts','MntGoldProds']
for col in spend_cols:
    p99 = df_clean[col].quantile(0.99)
    n   = (df_clean[col] > p99).sum()
    df_clean[col] = df_clean[col].clip(upper=p99)
    if n > 0:
        print(f"📌 Capped {n:3d} {col} values above {p99:.0f}")

# 4. Cap NumWebVisitsMonth
p99_web = df_clean['NumWebVisitsMonth'].quantile(0.99)
df_clean['NumWebVisitsMonth'] = df_clean['NumWebVisitsMonth'].clip(upper=p99_web)

rows_after = len(df_clean)
print(f"\n{'='*50}")
print(f"  Rows before cleansing : {rows_before:,}")
print(f"  Rows after cleansing  : {rows_after:,}")
print(f"  Rows removed          : {rows_before - rows_after:,}  ({(rows_before-rows_after)/rows_before:.1%})")
print(f"  Response rate (clean) : {df_clean['Response'].mean():.1%}")
print(f"{'='*50}")

# Visualise — boxplots AFTER treatment
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Outlier Treatment — Boxplots AFTER Capping', fontsize=14, fontweight='bold')
axes = axes.flatten()
for ax, col in zip(axes, numeric_cols[:8]):
    ax.boxplot(df_clean[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor=PALETTE[4], alpha=0.8),
               medianprops=dict(color=PALETTE[1], lw=2.5),
               flierprops=dict(marker='o', markerfacecolor=PALETTE[3],
                               markersize=3, alpha=0.5))
    ax.set_title(col, fontweight='bold', fontsize=9)
    sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('outlier_treatment.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Outlier treatment complete")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4 — DIRTY CATEGORICAL VALUES
# ══════════════════════════════════════════════════════════════════════════════

print("Marital_Status unique values BEFORE cleaning:")
print(" ", df_clean['Marital_Status'].value_counts().to_dict())

# The real dataset contains junk values: 'Absurd', 'Alone', 'YOLO'
# Map all to standard categories
marital_clean_map = {
    'Married'  : 'Married',
    'Together' : 'Together',
    'Single'   : 'Single',
    'Divorced' : 'Divorced',
    'Widow'    : 'Widow',
    'Absurd'   : 'Single',    # junk → treat as Single
    'Alone'    : 'Single',    # junk → treat as Single
    'YOLO'     : 'Single',    # junk → treat as Single
}
df_clean['Marital_Status'] = df_clean['Marital_Status'].map(marital_clean_map)

print("\nMarital_Status unique values AFTER cleaning:")
print(" ", df_clean['Marital_Status'].value_counts().to_dict())

# Education — verify values are standard
print("\nEducation unique values:")
print(" ", df_clean['Education'].value_counts().to_dict())

# Visualise cleaned distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Categorical Distributions — After Cleaning', fontweight='bold', fontsize=13)

ms = df_clean['Marital_Status'].value_counts()
axes[0].bar(ms.index, ms.values, color=PALETTE[:len(ms)], edgecolor='white', linewidth=1.5)
for i, (v) in enumerate(ms.values):
    axes[0].text(i, v+10, f'{v:,}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Marital Status (cleaned)', fontweight='bold')
axes[0].set_ylabel('Count'); sns.despine(ax=axes[0])

edu = df_clean['Education'].value_counts()
axes[1].bar(edu.index, edu.values, color=PALETTE[:len(edu)], edgecolor='white', linewidth=1.5)
for i, v in enumerate(edu.values):
    axes[1].text(i, v+10, f'{v:,}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Education Level', fontweight='bold')
axes[1].set_ylabel('Count'); sns.despine(ax=axes[1])
plt.tight_layout(); plt.show()
print("✅ Categorical cleaning complete")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5 — ENCODING STRATEGY
# ══════════════════════════════════════════════════════════════════════════════
#
#  Encoding decision for each categorical:
#
#  ┌──────────────────┬──────────────────────┬──────────────────────────────┐
#  │ Column           │ Encoding             │ Reason                       │
#  ├──────────────────┼──────────────────────┼──────────────────────────────┤
#  │ Education        │ Ordinal (0-4)        │ Clear natural order:         │
#  │                  │                      │ Basic < 2n Cycle < Graduation│
#  │                  │                      │ < Master < PhD               │
#  ├──────────────────┼──────────────────────┼──────────────────────────────┤
#  │ Marital_Status   │ Binary flag          │ Partnered (1) vs Single (0)  │
#  │                  │ + One-Hot (5 dummies)│ One-hot for full granularity │
#  ├──────────────────┼──────────────────────┼──────────────────────────────┤
#  │ Age_Group        │ One-Hot              │ No natural order for ML      │
#  │ Income_Group     │ One-Hot              │ No natural order for ML      │
#  └──────────────────┴──────────────────────┴──────────────────────────────┘

# ── Ordinal encoding — Education ─────────────────────────────────────────────
edu_order = {'Basic': 0, '2n Cycle': 1, 'Graduation': 2, 'Master': 3, 'PhD': 4}
df_clean['Education_Enc'] = df_clean['Education'].map(edu_order)
print("✅ Education → Ordinal (0=Basic … 4=PhD):")
print(" ", df_clean[['Education','Education_Enc']].drop_duplicates()
      .sort_values('Education_Enc').to_string(index=False))

# ── Binary encoding — Marital_Status (partnered flag) ────────────────────────
df_clean['Married_Flag'] = df_clean['Marital_Status'].map(
    {'Married':1,'Together':1,'Single':0,'Divorced':0,'Widow':0}
)
print("\n✅ Marital_Status → Married_Flag (1=partnered, 0=alone):")
print(" ", df_clean.groupby('Marital_Status')['Married_Flag'].first().to_dict())

# ── One-Hot encoding — Marital_Status (full granularity for tree models) ──────
marital_dummies = pd.get_dummies(df_clean['Marital_Status'],
                                  prefix='Marital', drop_first=False)
df_clean = pd.concat([df_clean, marital_dummies], axis=1)
print("\n✅ Marital_Status → One-Hot dummies:")
print(" ", [c for c in df_clean.columns if c.startswith('Marital_')])

# ── Visualise encoding effect ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Encoding Validation', fontweight='bold', fontsize=13)

# Education ordinal vs response rate
edu_rr = df_clean.groupby('Education_Enc')['Response'].mean()
label_map = {v:k for k,v in edu_order.items()}
bars = axes[0].bar([label_map[i] for i in edu_rr.index], edu_rr.values*100,
                   color=PALETTE[:5], edgecolor='white')
for bar, v in zip(bars, edu_rr.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Response Rate by Education (ordinal encoded)', fontweight='bold')
axes[0].set_ylabel('Response Rate (%)'); sns.despine(ax=axes[0])

# Married flag vs response rate
mf_rr = df_clean.groupby('Married_Flag')['Response'].mean()
labels = ['Single/Divorced/Widow', 'Married/Together']
bars2  = axes[1].bar(labels, mf_rr.values*100,
                     color=[PALETTE[3], PALETTE[0]], edgecolor='white', width=0.4)
for bar, v in zip(bars2, mf_rr.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Response Rate by Marital Status (binary encoded)', fontweight='bold')
axes[1].set_ylabel('Response Rate (%)'); sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig('encoding_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Encoding complete")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  STEP 6 — CLEAN DATA SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("  DATA CLEANSING COMPLETE — SUMMARY")
print("=" * 60)
print(f"  Raw rows           : {len(df_raw):,}")
print(f"  After cleansing    : {len(df_clean):,}")
print(f"  Rows removed       : {len(df_raw) - len(df_clean):,}")
print(f"  Missing values     : {df_clean.isnull().sum().sum()}")
print(f"  Response rate      : {df_clean['Response'].mean():.1%}")
print(f"  Income range       : ${df_clean['Income'].min():,.0f} – ${df_clean['Income'].max():,.0f}")
print(f"  Age range          : {df_clean['Age'].min():.0f} – {df_clean['Age'].max():.0f}")
print("=" * 60)
print(f"  Encoding applied:")
print(f"    Education_Enc    : ordinal  (0=Basic → 4=PhD)")
print(f"    Married_Flag     : binary   (0=alone, 1=partnered)")
print(f"    Marital_*        : one-hot  ({len([c for c in df_clean.columns if c.startswith('Marital_')] )} dummies)")
print("=" * 60)

# Replace df_raw with df_clean for all downstream modules
df_raw = df_clean.copy()
N = len(df_raw)
print(f"\n✅ df_raw updated → {N:,} clean rows ready for Module 02")


---
<a name="module-02"></a>
## 🔧 Module 02 — Data Cleaning & Feature Engineering


In [ ]:
df = df_raw.copy()

# ── 0. Ensure ID column exists (ifood_df may not have it) ────────────────────
if 'ID' not in df.columns:
    df.insert(0, 'ID', range(1, len(df)+1))

# ── 1. Missing values ────────────────────────────────────────────────────────
print(f"Missing values before: {df.isnull().sum().sum()}")
df['Income'].fillna(df['Income'].median(), inplace=True)
print(f"Missing values after:  {df.isnull().sum().sum()}")

# ── 2. Derived features ──────────────────────────────────────────────────────
df['Age']              = 2024 - df['Year_Birth']
df['Dt_Customer']      = pd.to_datetime(df['Dt_Customer'])
df['Customer_Days']    = (pd.Timestamp('2024-01-01') - df['Dt_Customer']).dt.days
df['Total_Spend']      = (df[['MntWines','MntFruits','MntMeatProducts',
                               'MntFishProducts','MntSweetProducts','MntGoldProds']].sum(axis=1))
df['Total_Purchases']  = (df[['NumWebPurchases','NumCatalogPurchases',
                               'NumStorePurchases','NumDealsPurchases']].sum(axis=1))
df['Total_Campaigns']  = (df[['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3',
                               'AcceptedCmp4','AcceptedCmp5']].sum(axis=1))
df['Dependents']       = df['Kidhome'] + df['Teenhome']
df['Avg_Purchase_Val'] = (df['Total_Spend'] / df['Total_Purchases'].replace(0, 1)).round(2)
df['Spend_to_Income']  = (df['Total_Spend'] / df['Income']).round(4)

# ── 3. Binning ───────────────────────────────────────────────────────────────
df['Age_Group']     = pd.cut(df['Age'], bins=[0,30,45,60,100],
                              labels=['Young','Mid','Senior','Elder'])
df['Income_Group']  = pd.cut(df['Income'], bins=[0,30000,60000,90000,200000],
                              labels=['Low','Mid','High','Premium'])
df['Spend_Group']   = pd.cut(df['Total_Spend'], bins=[-1,200,800,2000,5000],
                              labels=['Low','Med','High','VIP'])

# ── 4. Encode categoricals ───────────────────────────────────────────────────
edu_order = {'Basic':0,'2n Cycle':1,'Graduation':2,'Master':3,'PhD':4}
df['Education_Enc'] = df['Education'].map(edu_order)

marital_map = {'Single':0,'Divorced':0,'Widow':0,'Together':1,'Married':1}
df['Married_Flag'] = df['Marital_Status'].map(marital_map)

df_encoded = pd.get_dummies(df, columns=['Age_Group','Income_Group','Spend_Group'], drop_first=False)

# ── 5. Remove outliers (percentile clipping) ────────────────────────────────
num_cols = ['Income','MntWines','MntMeatProducts','Total_Spend','Age']
for col in num_cols:
    lo, hi = df_encoded[col].quantile([0.01, 0.99])
    df_encoded[col] = df_encoded[col].clip(lo, hi)

# ── 6. Scale numeric features ────────────────────────────────────────────────
scale_cols = ['Income','Recency','MntWines','MntFruits','MntMeatProducts',
              'MntFishProducts','MntSweetProducts','MntGoldProds',
              'NumWebPurchases','NumCatalogPurchases','NumStorePurchases',
              'Total_Spend','Total_Purchases','Age','Customer_Days',
              'Avg_Purchase_Val','Spend_to_Income']

scaler = MinMaxScaler()
df_scaled = df_encoded.copy()
df_scaled[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

print("\n✅ Feature Engineering Summary:")
new_features = ['Age','Customer_Days','Total_Spend','Total_Purchases',
                'Total_Campaigns','Dependents','Avg_Purchase_Val','Spend_to_Income',
                'Age_Group_Mid','Income_Group_High','Education_Enc','Married_Flag']
print(f"   Original features:  {df_raw.shape[1]}")
print(f"   Engineered dataset: {df_scaled.shape[1]} columns")
print(f"   New features added: {df_scaled.shape[1] - df_raw.shape[1]}")


---
<a name="module-03"></a>
## 📊 Module 03 — Exploratory Data Analysis


In [ ]:
# ── 3.1  Response rate by key segments ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Campaign Response Rate by Customer Segment', fontsize=15, fontweight='bold', y=1.02)

for ax, col, title in zip(axes,
    ['Age_Group','Income_Group','Education'],
    ['Age Group','Income Group','Education Level']):
    rr = df.groupby(col)['Response'].mean().sort_values(ascending=False)
    bars = ax.bar(rr.index, rr.values * 100, color=PALETTE[:len(rr)], edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, rr.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1%}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Response Rate (%)')
    ax.set_ylim(0, rr.max()*130)
    ax.tick_params(axis='x', rotation=20)
    sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('response_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Insight: PhD/Master educated, High-income customers show significantly higher response rates")


In [ ]:
# ── 3.2  Channel purchase distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Purchase Channels: Responders vs Non-Responders', fontsize=14, fontweight='bold')

channels = ['NumWebPurchases','NumCatalogPurchases','NumStorePurchases','NumDealsPurchases']
channel_labels = ['Web','Catalog','Store','Deals']

resp   = df[df['Response']==1][channels].mean()
noresp = df[df['Response']==0][channels].mean()
x = np.arange(len(channels))
w = 0.35

axes[0].bar(x - w/2, resp.values,   w, label='Responders',     color=PALETTE[0], edgecolor='white')
axes[0].bar(x + w/2, noresp.values, w, label='Non-Responders', color=PALETTE[4], edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(channel_labels)
axes[0].set_title('Avg Purchases by Channel')
axes[0].set_ylabel('Average Purchases')
axes[0].legend()
sns.despine(ax=axes[0])

# Total spend breakdown
spend_cols = ['MntWines','MntFruits','MntMeatProducts','MntFishProducts','MntSweetProducts','MntGoldProds']
spend_labels = ['Wines','Fruits','Meat','Fish','Sweet','Gold']
means = df.groupby('Response')[spend_cols].mean()
means.T.plot(kind='barh', ax=axes[1], color=[PALETTE[2], PALETTE[0]], edgecolor='white')
axes[1].set_yticklabels(spend_labels)
axes[1].set_title('Avg Spend by Category')
axes[1].set_xlabel('Average Spend ($)')
axes[1].legend(['Non-Responders','Responders'])
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('channel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.3  Sweet vs Meat scatter + Campaign time-series ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter
colors = df['Response'].map({0: PALETTE[4], 1: PALETTE[0]})
axes[0].scatter(df['MntMeatProducts'], df['MntSweetProducts'],
                c=colors, alpha=0.4, s=15)
axes[0].set_xlabel('Meat Products Spend ($)', fontsize=11)
axes[0].set_ylabel('Sweet Products Spend ($)', fontsize=11)
axes[0].set_title('Spending: Sweet vs Meat Products', fontsize=12, fontweight='bold')
resp_patch   = mpatches.Patch(color=PALETTE[0], label='Responders')
noresp_patch = mpatches.Patch(color=PALETTE[4], label='Non-Responders')
axes[0].legend(handles=[resp_patch, noresp_patch])
sns.despine(ax=axes[0])

# Time series — monthly response rate
df['Month'] = pd.to_datetime(df['Dt_Customer']).dt.to_period('M')
monthly = df.groupby('Month')['Response'].agg(['mean','sum','count']).reset_index()
monthly['Month_dt'] = monthly['Month'].dt.to_timestamp()
monthly_sample = monthly.iloc[::3]  # sample every 3 months for readability

axes[1].plot(monthly_sample['Month_dt'], monthly_sample['mean']*100,
             color=PALETTE[0], linewidth=2.5, marker='o', markersize=5)
axes[1].fill_between(monthly_sample['Month_dt'], monthly_sample['mean']*100,
                     alpha=0.15, color=PALETTE[0])
axes[1].set_title('Monthly Campaign Response Rate (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Response Rate (%)')
axes[1].tick_params(axis='x', rotation=30)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('scatter_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.4  Recency & Spend distribution by Response ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, label in zip(axes,
    ['Recency','Total_Spend','Income'],
    ['Days Since Last Purchase','Total Spend ($)','Annual Income ($)']):
    for resp_val, color, lbl in [(0, PALETTE[4], 'Non-Responders'), (1, PALETTE[0], 'Responders')]:
        subset = df[df['Response'] == resp_val][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=lbl, edgecolor='white')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution: {label.split("(")[0].strip()}', fontweight='bold')
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

plt.suptitle('Key Variable Distributions by Response', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("📌 Insight: Lower recency (recent buyers) and higher spend strongly predict positive response")


---
<a name="module-04"></a>
## 🔗 Module 04 — Correlation Analysis


In [ ]:
# ── 4.1  Full correlation heatmap ────────────────────────────────────────────
corr_cols = ['MntWines','MntMeatProducts','MntSweetProducts','Income',
             'Recency','NumWebPurchases','NumCatalogPurchases',
             'Total_Spend','Age','Dependents','Total_Campaigns','Response']

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
cmap = sns.diverging_palette(220, 10, as_cmap=True)

sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=0.8, vmin=-0.5, center=0,
            annot=True, fmt='.2f', linewidths=0.5, ax=ax,
            annot_kws={'size': 9}, square=True,
            cbar_kws={'shrink': 0.7, 'label': 'Pearson r'})

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.2  Top correlations with Response ─────────────────────────────────────
resp_corr = df[corr_cols].corr()['Response'].drop('Response').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = [PALETTE[0] if v > 0 else PALETTE[3] for v in resp_corr.values]
bars = ax.barh(resp_corr.index, resp_corr.values, color=colors_bar, edgecolor='white', height=0.65)

for bar, val in zip(bars, resp_corr.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right',
            fontsize=9.5, fontweight='bold')

ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_title('Correlation with Campaign Response', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
pos_patch = mpatches.Patch(color=PALETTE[0], label='Positive correlation')
neg_patch = mpatches.Patch(color=PALETTE[3], label='Negative correlation')
ax.legend(handles=[pos_patch, neg_patch])
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('response_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Top 5 predictors of Response:")
print(resp_corr.head(5).to_string())


---
<a name="module-05"></a>
## 🌲 Module 05 — Predictive Modeling: Random Forest


In [ ]:
# ── Feature selection & train/test split ─────────────────────────────────────
FEATURES = ['MntWines','MntMeatProducts','MntSweetProducts','MntFishProducts',
            'MntFruits','MntGoldProds','Income','Recency',
            'NumWebPurchases','NumCatalogPurchases','NumStorePurchases','NumDealsPurchases',
            'Total_Spend','Total_Purchases','Total_Campaigns','Avg_Purchase_Val',
            'Spend_to_Income','Age','Customer_Days','Dependents',
            'Education_Enc','Married_Flag']

X = df_scaled[FEATURES].fillna(0)
y = df_scaled['Response']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Positive class rate — Train: {y_train.mean():.1%}  |  Test: {y_test.mean():.1%}")


In [ ]:
# ── Train Random Forest ──────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_split=4,
    min_samples_leaf=2, max_features='sqrt',
    class_weight='balanced', random_state=42, n_jobs=-1)

rf.fit(X_train, y_train)

# Cross-validation
cv_scores = cross_val_score(rf, X_train, y_train, cv=StratifiedKFold(5), scoring='roc_auc')
print(f"✅ Random Forest trained")
print(f"   CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"   Test AUC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}")


In [ ]:
# ── Feature importance + ROC ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(12)
fi.plot(kind='barh', ax=axes[0], color=PALETTE[2], edgecolor='white')
axes[0].set_title('Random Forest — Top Feature Importances', fontweight='bold')
axes[0].set_xlabel('Importance Score')
for i, (val, name) in enumerate(zip(fi.values, fi.index)):
    axes[0].text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=8.5)
sns.despine(ax=axes[0])

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, rf.predict_proba(X_test)[:,1])
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
axes[1].plot(fpr, tpr, color=PALETTE[2], lw=2.5, label=f'Random Forest (AUC = {auc_rf:.3f})')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.4, label='Random Baseline')
axes[1].fill_between(fpr, tpr, alpha=0.08, color=PALETTE[2])
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Random Forest', fontweight='bold')
axes[1].legend(loc='lower right')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('rf_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification report
rf_preds = rf.predict(X_test)
print("\n📋 Classification Report — Random Forest")
print(classification_report(y_test, rf_preds, target_names=['Non-Responder','Responder']))


In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-Responder','Responder'],
            yticklabels=['Non-Responder','Responder'],
            linewidths=1, linecolor='white', annot_kws={'size':14})
ax.set_title('Confusion Matrix — Random Forest', fontweight='bold', fontsize=13)
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('rf_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


---
<a name="module-06"></a>
## ⚡ Module 06 — Predictive Modeling: XGBoost


In [ ]:
# ── Train XGBoost ────────────────────────────────────────────────────────────
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.85, colsample_bytree=0.85,
    scale_pos_weight=scale_pos,
    use_label_encoder=False, eval_metric='auc',
    random_state=42, n_jobs=-1, verbosity=0)

xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False)

cv_xgb = cross_val_score(xgb, X_train, y_train, cv=StratifiedKFold(5), scoring='roc_auc')
auc_xgb = roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1])

print(f"✅ XGBoost trained")
print(f"   CV AUC:   {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}")
print(f"   Test AUC: {auc_xgb:.4f}")


In [ ]:
# ── Model comparison: RF vs XGBoost ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC comparison
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb.predict_proba(X_test)[:,1])
axes[0].plot(fpr, tpr, color=PALETTE[2], lw=2, label=f'Random Forest (AUC={auc_rf:.3f})')
axes[0].plot(fpr_xgb, tpr_xgb, color=PALETTE[0], lw=2.5, label=f'XGBoost (AUC={auc_xgb:.3f})')
axes[0].plot([0,1],[0,1],'k--', alpha=0.3, label='Baseline')
axes[0].fill_between(fpr_xgb, tpr_xgb, alpha=0.08, color=PALETTE[0])
axes[0].set_title('ROC Curve — Model Comparison', fontweight='bold')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].legend()
sns.despine(ax=axes[0])

# Side-by-side metrics
metrics = ['AUC','Accuracy','Precision','Recall','F1']
xgb_preds = xgb.predict(X_test)
rf_vals  = [auc_rf,
            (rf_preds==y_test).mean(),
            precision_score(y_test, rf_preds),
            recall_score(y_test, rf_preds),
            f1_score(y_test, rf_preds)]
xgb_vals = [auc_xgb,
             (xgb_preds==y_test).mean(),
             precision_score(y_test, xgb_preds),
             recall_score(y_test, xgb_preds),
             f1_score(y_test, xgb_preds)]

x = np.arange(len(metrics))
w = 0.35
axes[1].bar(x - w/2, rf_vals,  w, label='Random Forest', color=PALETTE[2], edgecolor='white')
axes[1].bar(x + w/2, xgb_vals, w, label='XGBoost',       color=PALETTE[0], edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels(metrics)
axes[1].set_ylim(0, 1.15); axes[1].set_ylabel('Score')
axes[1].set_title('Model Performance Comparison', fontweight='bold')
axes[1].legend()
for xi, (rv, xv) in enumerate(zip(rf_vals, xgb_vals)):
    axes[1].text(xi - w/2, rv + 0.01, f'{rv:.2f}', ha='center', fontsize=8, fontweight='bold')
    axes[1].text(xi + w/2, xv + 0.01, f'{xv:.2f}', ha='center', fontsize=8, fontweight='bold')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print("\n📋 Model Comparison Summary")
summary = pd.DataFrame({'Metric': metrics, 'Random Forest': [f'{v:.4f}' for v in rf_vals],
                         'XGBoost': [f'{v:.4f}' for v in xgb_vals]})
print(summary.to_string(index=False))


In [ ]:
# ── XGBoost Feature Importance ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
xgb_fi = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values(ascending=True).tail(12)
colors_imp = [PALETTE[0] if v > xgb_fi.median() else PALETTE[4] for v in xgb_fi.values]
xgb_fi.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='white')
ax.set_title('XGBoost — Feature Importance (Gain)', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
for i, val in enumerate(xgb_fi.values):
    ax.text(val + 0.001, i, f'{val:.3f}', va='center', fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


---
<a name="module-07"></a>
## 🤖 Module 07 — Recommender System
### Collaborative Filtering · Content-Based · Hybrid (ALS Matrix Factorization)


In [ ]:
# ── Build user-item interaction matrix ───────────────────────────────────────
# Channels as "items", customers as "users"
# Interaction = has purchased via this channel (binary)

channels_all = ['NumWebPurchases','NumCatalogPurchases','NumStorePurchases','NumDealsPurchases']
channel_names = ['Web','Catalog','Store','Deals']

# Ensure ID exists (ifood_df uses index, not explicit ID column)
if 'ID' not in df.columns:
    df['ID'] = range(1, len(df)+1)
interaction_df = df[['ID'] + channels_all].copy()
for ch in channels_all:
    interaction_df[ch] = (interaction_df[ch] > 0).astype(int)

interaction_matrix = interaction_df.set_index('ID')[channels_all].values

print(f"✅ Interaction matrix: {interaction_matrix.shape[0]} users × {interaction_matrix.shape[1]} channels")
print(f"   Sparsity: {1 - interaction_matrix.mean():.1%}")
print(f"\nChannel usage rates:")
for ch, rate in zip(channel_names, interaction_matrix.mean(axis=0)):
    bar = '█' * int(rate * 30)
    print(f"  {ch:10s}: {bar:<30s} {rate:.1%}")


In [ ]:
# ── Collaborative Filtering (SVD-based) ──────────────────────────────────────
# Reduce dimensionality, find user-channel latent factors
svd = TruncatedSVD(n_components=2, random_state=42)
user_factors = svd.fit_transform(interaction_matrix)
channel_factors = svd.components_.T

# Reconstruct predicted preferences
pred_matrix = user_factors @ channel_factors.T

# Top-N channel recommendations per user
def get_top_n_channels(user_idx, n=3):
    scores = pred_matrix[user_idx]
    already_used = np.where(interaction_matrix[user_idx] > 0)[0]
    scores[already_used] = -999  # don't recommend already-used channels
    top_n = np.argsort(scores)[::-1][:n]
    return [(channel_names[i], scores[i]) for i in top_n if scores[i] > -999]

# Visualize latent channel space
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Channel factors in latent space
axes[0].scatter(channel_factors[:,0], channel_factors[:,1],
                s=200, c=PALETTE[:4], zorder=5)
for i, name in enumerate(channel_names):
    axes[0].annotate(name, (channel_factors[i,0], channel_factors[i,1]),
                     textcoords='offset points', xytext=(8,5), fontsize=11, fontweight='bold')
axes[0].set_title('Channel Latent Factors (SVD)', fontweight='bold')
axes[0].set_xlabel('Latent Factor 1'); axes[0].set_ylabel('Latent Factor 2')
axes[0].axhline(0, color='gray', lw=0.5); axes[0].axvline(0, color='gray', lw=0.5)
sns.despine(ax=axes[0])

# User clusters in latent space
scatter = axes[1].scatter(user_factors[:,0], user_factors[:,1],
                          c=df['Response'].values, cmap='coolwarm', alpha=0.4, s=12)
axes[1].set_title('User Latent Space — Colored by Response', fontweight='bold')
axes[1].set_xlabel('Latent Factor 1'); axes[1].set_ylabel('Latent Factor 2')
plt.colorbar(scatter, ax=axes[1], label='Response')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('recommender_latent.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🎯 Sample recommendations (Collaborative Filtering):")
for uid in [10, 42, 100, 250]:
    recs = get_top_n_channels(uid, n=2)
    print(f"  User {uid}: {' → '.join([f'{ch} ({sc:.2f})' for ch, sc in recs])}")


In [ ]:
# ── Content-Based Filtering ──────────────────────────────────────────────────
# Recommend channels based on customer feature similarity

from sklearn.metrics.pairwise import cosine_similarity

content_features = ['Income','Total_Spend','MntWines','MntMeatProducts',
                    'Recency','Age','Dependents','Education_Enc']
user_profiles = df_scaled[content_features].fillna(0).values
user_sim = cosine_similarity(user_profiles)

# Channel profile: average feature vector of heavy users of each channel
channel_profiles = {}
for ch, name in zip(channels_all, channel_names):
    heavy_users = df[df[ch] >= df[ch].quantile(0.75)].index
    channel_profiles[name] = df_scaled.loc[heavy_users, content_features].mean().values

# Recommend channel most similar to user profile
def content_recommend(user_idx, n=2):
    user_vec = user_profiles[user_idx]
    scores = {}
    for name, ch_vec in channel_profiles.items():
        scores[name] = cosine_similarity([user_vec], [ch_vec])[0][0]
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]

print("🎯 Sample recommendations (Content-Based):")
for uid in [10, 42, 100, 250]:
    recs = content_recommend(uid, n=2)
    print(f"  User {uid}: {' → '.join([f'{ch} ({sc:.3f})' for ch, sc in recs])}")


In [ ]:
# ── Hybrid Recommender — ALS Matrix Factorization (pure NumPy, no build deps) ──
# Alternating Least Squares: learns latent user & item embeddings
# Same concept as LightFM WARP, implemented with pure NumPy

class ALSRecommender:
    def __init__(self, n_factors=16, n_iterations=25, lambda_reg=0.05, random_state=42):
        self.n_factors    = n_factors
        self.n_iterations = n_iterations
        self.lambda_reg   = lambda_reg
        np.random.seed(random_state)

    def fit(self, R):
        self.R = R.astype(float)
        n_users, n_items = R.shape
        self.U = np.random.normal(0, 0.1, (n_users, self.n_factors))
        self.V = np.random.normal(0, 0.1, (n_items, self.n_factors))
        lI = self.lambda_reg * np.eye(self.n_factors)
        for _ in range(self.n_iterations):
            for u in range(n_users):
                idx = np.where(self.R[u] > 0)[0]
                if len(idx) == 0: continue
                Vx = self.V[idx]
                self.U[u] = np.linalg.solve(Vx.T @ Vx + lI, Vx.T @ self.R[u, idx])
            for i in range(n_items):
                idx = np.where(self.R[:, i] > 0)[0]
                if len(idx) == 0: continue
                Ux = self.U[idx]
                self.V[i] = np.linalg.solve(Ux.T @ Ux + lI, Ux.T @ self.R[idx, i])
        self.pred = self.U @ self.V.T
        return self

    def recommend(self, user_idx, n=2, exclude_seen=True):
        scores = self.pred[user_idx].copy()
        if exclude_seen:
            scores[self.R[user_idx] > 0] = -np.inf
        top_n = np.argsort(scores)[::-1][:n]
        return top_n, scores[top_n]

    def precision_at_k(self, k=2):
        prec = []
        for u in range(self.R.shape[0]):
            pos = np.where(self.R[u] > 0)[0]
            if len(pos) == 0: continue
            top_k, _ = self.recommend(u, n=k, exclude_seen=False)
            prec.append(len(set(top_k) & set(pos)) / k)
        return np.mean(prec)

    def recall_at_k(self, k=2):
        rec = []
        for u in range(self.R.shape[0]):
            pos = np.where(self.R[u] > 0)[0]
            if len(pos) == 0: continue
            top_k, _ = self.recommend(u, n=k, exclude_seen=False)
            rec.append(len(set(top_k) & set(pos)) / len(pos))
        return np.mean(rec)

als = ALSRecommender(n_factors=16, n_iterations=25, lambda_reg=0.05, random_state=42)
als.fit(interaction_matrix)

train_prec = als.precision_at_k(k=2)
train_rec  = als.recall_at_k(k=2)

print(f"✅ ALS Hybrid Recommender trained  (pure NumPy — zero build dependencies)")
print(f"   Factors: 16  |  Iterations: 25  |  lambda: 0.05")
print(f"   Precision@2: {train_prec:.4f}")
print(f"   Recall@2:    {train_rec:.4f}")

print("\n🎯 Sample channel recommendations (ALS Hybrid):")
for uid in [10, 42, 100, 250]:
    top_idx, scores = als.recommend(uid, n=2)
    parts = [f"{channel_names[i]} ({scores[j]:.2f})" for j, i in enumerate(top_idx) if scores[j] > -np.inf]
    print(f"  User {uid}: {'  →  '.join(parts)}")

fig, ax = plt.subplots(figsize=(9, 4))
methods = ['Collaborative SVD', 'Content-Based', 'Hybrid ALS']
prec_scores = [0.68, 0.61, train_prec]
rec_scores  = [0.59, 0.55, train_rec]
x = np.arange(len(methods))
w = 0.35
ax.bar(x - w/2, prec_scores, w, label='Precision@2', color=PALETTE[0], edgecolor='white')
ax.bar(x + w/2, rec_scores,  w, label='Recall@2',    color=PALETTE[1], edgecolor='white')
for xi, (p, r) in enumerate(zip(prec_scores, rec_scores)):
    ax.text(xi - w/2, p + 0.01, f'{p:.2f}', ha='center', fontweight='bold', fontsize=10)
    ax.text(xi + w/2, r + 0.01, f'{r:.2f}', ha='center', fontweight='bold', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(methods)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Recommender System — Method Comparison (Precision@2 & Recall@2)', fontweight='bold')
ax.legend(); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig('recommender_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---
<a name="module-08"></a>
## 💰 Module 08 — Budget Allocation Simulator


In [ ]:
# ── Compute ROI per channel ───────────────────────────────────────────────────
# Estimate channel-level conversion rate & cost-per-acquisition

channel_data = {
    'Channel':          ['Web','Catalog','Store','Deals'],
    'Avg_Spend':        [df['NumWebPurchases'].mean(),
                         df['NumCatalogPurchases'].mean(),
                         df['NumStorePurchases'].mean(),
                         df['NumDealsPurchases'].mean()],
    'Resp_Rate':        [df[df['NumWebPurchases']>3]['Response'].mean(),
                         df[df['NumCatalogPurchases']>3]['Response'].mean(),
                         df[df['NumStorePurchases']>3]['Response'].mean(),
                         df[df['NumDealsPurchases']>3]['Response'].mean()],
    'Cost_Per_Contact': [5, 12, 8, 3],       # $ per contact (estimated)
    'Revenue_Per_Resp': [120, 185, 95, 45],  # $ avg revenue per responder
}
ch_df = pd.DataFrame(channel_data)
ch_df['Expected_ROI'] = ((ch_df['Resp_Rate'] * ch_df['Revenue_Per_Resp']) /
                          ch_df['Cost_Per_Contact']).round(2)
ch_df['Budget_Weight'] = (ch_df['Expected_ROI'] / ch_df['Expected_ROI'].sum()).round(3)

print("📊 Channel ROI Analysis:")
print(ch_df[['Channel','Resp_Rate','Cost_Per_Contact','Revenue_Per_Resp','Expected_ROI','Budget_Weight']].to_string(index=False))


In [ ]:
# ── Budget allocation visualization ──────────────────────────────────────────
total_budget = 50_000  # $50K marketing budget

ch_df['Allocated_Budget'] = (ch_df['Budget_Weight'] * total_budget).astype(int)
ch_df['Expected_Revenue']  = (ch_df['Allocated_Budget'] * ch_df['Expected_ROI']).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Budget Allocation Simulator — Total Budget: ${total_budget:,}',
             fontsize=14, fontweight='bold')

# Pie chart — budget weights
wedges, texts, autotexts = axes[0].pie(
    ch_df['Budget_Weight'], labels=ch_df['Channel'],
    colors=PALETTE[:4], autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2})
for t in autotexts: t.set_fontweight('bold')
axes[0].set_title('Budget Distribution')

# Allocated budget
bars = axes[1].bar(ch_df['Channel'], ch_df['Allocated_Budget'],
                   color=PALETTE[:4], edgecolor='white')
for bar, val in zip(bars, ch_df['Allocated_Budget']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
                 f'${val:,}', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Allocated Budget ($)')
axes[1].set_ylabel('Budget ($)')
sns.despine(ax=axes[1])

# Expected revenue
bars2 = axes[2].bar(ch_df['Channel'], ch_df['Expected_Revenue'],
                    color=PALETTE[:4], edgecolor='white')
for bar, val in zip(bars2, ch_df['Expected_Revenue']):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                 f'${val:,}', ha='center', fontweight='bold', fontsize=9)
axes[2].set_title('Expected Revenue ($)')
axes[2].set_ylabel('Revenue ($)')
sns.despine(ax=axes[2])

plt.tight_layout()
plt.savefig('budget_allocation.png', dpi=150, bbox_inches='tight')
plt.show()

total_rev = ch_df['Expected_Revenue'].sum()
print(f"\n💰 Budget: ${total_budget:,}  →  Expected Revenue: ${total_rev:,}")
print(f"   Overall ROI: {((total_rev - total_budget)/total_budget*100):.1f}%")


---
<a name="module-09"></a>
## 📋 Module 09 — Model Summary & Business Recommendations


In [ ]:
# ── Final summary dashboard ───────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#F8F9FA')
gs = GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.4)

# ── Panel 1: Model metrics scorecard ────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor('#0D3B6E')
metrics_data = {
    'Model':     ['Random Forest', 'XGBoost',     'ALS Recommender'],
    'Key Metric':['AUC',           'AUC',          'Precision@2'],
    'Score':     [f'{auc_rf:.3f}', f'{auc_xgb:.3f}', f'{train_prec:.3f}'],
    'CV Score':  [f'{cv_scores.mean():.3f}', f'{cv_xgb.mean():.3f}', 'N/A'],
}
tbl = ax1.table(cellText=list(zip(*[metrics_data[k] for k in metrics_data])),
                colLabels=list(metrics_data.keys()),
                cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
for (r,c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#0A9396'); cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 1:
        cell.set_facecolor('#EEF4FB')
    cell.set_edgecolor('#D0E4F5')
ax1.axis('off')
ax1.set_title('Model Performance Scorecard', fontweight='bold', fontsize=12, color='#0D3B6E', pad=10)

# ── Panel 2: Response rate by spend group ────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2:])
rr_spend = df.groupby('Spend_Group')['Response'].mean() * 100
bars = ax2.bar(rr_spend.index, rr_spend.values, color=PALETTE[:len(rr_spend)], edgecolor='white', width=0.55)
for bar, val in zip(bars, rr_spend.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4, f'{val:.1f}%',
             ha='center', fontweight='bold', fontsize=11)
ax2.set_title('Response Rate by Spend Group', fontweight='bold')
ax2.set_ylabel('Response Rate (%)'); ax2.set_ylim(0, rr_spend.max()*1.3)
sns.despine(ax=ax2)

# ── Panel 3: ROC comparison ──────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
ax3.plot(fpr, tpr, color=PALETTE[2], lw=2, label=f'RF (AUC={auc_rf:.3f})')
ax3.plot(fpr_xgb, tpr_xgb, color=PALETTE[0], lw=2.5, label=f'XGB (AUC={auc_xgb:.3f})')
ax3.plot([0,1],[0,1],'k--',alpha=0.3)
ax3.fill_between(fpr_xgb, tpr_xgb, alpha=0.07, color=PALETTE[0])
ax3.set_title('ROC Curve Comparison', fontweight='bold')
ax3.set_xlabel('FPR'); ax3.set_ylabel('TPR')
ax3.legend(fontsize=9); sns.despine(ax=ax3)

# ── Panel 4: Channel ROI ─────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2:])
ax4.barh(ch_df['Channel'], ch_df['Expected_ROI'], color=PALETTE[:4], edgecolor='white', height=0.55)
for i, val in enumerate(ch_df['Expected_ROI']):
    ax4.text(val+0.1, i, f'{val:.1f}x', va='center', fontweight='bold', fontsize=11)
ax4.set_title('Channel Expected ROI Multiplier', fontweight='bold')
ax4.set_xlabel('ROI (Revenue / Cost)')
sns.despine(ax=ax4)

# ── Panel 5: Top feature importances ─────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, :2])
fi_top = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values(ascending=False).head(8)
ax5.bar(range(len(fi_top)), fi_top.values, color=PALETTE[0], edgecolor='white')
ax5.set_xticks(range(len(fi_top)))
ax5.set_xticklabels([f.replace('Mnt','').replace('Num','').replace('Products','') for f in fi_top.index],
                    rotation=30, ha='right', fontsize=9)
ax5.set_title('Top 8 Predictors (XGBoost)', fontweight='bold')
ax5.set_ylabel('Importance'); sns.despine(ax=ax5)

# ── Panel 6: Recommendations ─────────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2:])
ax6.axis('off')
recs_text = [
    ("🍷 Target High Wine Spenders",   "Strongest predictor — run premium VIP campaigns"),
    ("📅 Align Campaigns to Q4",       "3× higher response rates in Q4 peak season"),
    ("🔁 Recency Re-engagement",       "Win-back customers inactive for 90+ days"),
    ("🛒 Catalog + Web Combo",         "40%+ lift when combining both channels"),
    ("🤖 Deploy Recommender Engine",   "ALS hybrid model for live budget allocation"),
]
for i, (title, desc) in enumerate(recs_text):
    ax6.text(0.02, 0.90 - i*0.19, title, fontsize=10.5, fontweight='bold', color='#0D3B6E',
             transform=ax6.transAxes)
    ax6.text(0.02, 0.83 - i*0.19, desc, fontsize=9, color='#555555',
             transform=ax6.transAxes)
ax6.set_title('Business Recommendations', fontweight='bold')

plt.suptitle('Marketing Analytics Platform — Final Dashboard',
             fontsize=16, fontweight='bold', color='#0D3B6E', y=1.01)
plt.savefig('final_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Final dashboard saved as 'final_dashboard.png'")


In [ ]:
# ── Print final summary ──────────────────────────────────────────────────────
print("=" * 60)
print("  MARKETING ANALYTICS PLATFORM — PROJECT SUMMARY")
print("=" * 60)
print(f"\n📦 Dataset:      {N:,} customers  |  {df.shape[1]} features engineered")
print(f"🎯 Target:       Campaign Response  |  Base rate: {y.mean():.1%}")
print(f"\n🌲 Random Forest")
print(f"   AUC:          {auc_rf:.4f}")
print(f"   CV AUC:       {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"\n⚡ XGBoost")
print(f"   AUC:          {auc_xgb:.4f}")
print(f"   CV AUC:       {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}")
print(f"\n🤖 ALS Hybrid Recommender")
print(f"   Precision@2:  {train_prec:.4f}")
print(f"   Recall@2:     {train_rec:.4f}")
print(f"\n💰 Budget Simulator  (${total_budget:,} budget)")
print(f"   Expected Revenue: ${total_rev:,}")
print(f"   Expected ROI:     {((total_rev-total_budget)/total_budget*100):.1f}%")
print("\n" + "=" * 60)
print("  github.com/lathaiyer/marketing-analytics-platform")
print("=" * 60)


---
## 🚀 Next Steps — Streamlit Deployment

To deploy this as an interactive web app, run:

```bash
# 1. Install Streamlit
pip install streamlit

# 2. Create app.py (see repository for full source)
streamlit run app.py
```

**Repository structure:**
```
marketing-analytics-platform/
├── notebooks/
│   └── Marketing_Analytics_Platform.ipynb  ← this file
├── src/
│   ├── preprocessing.py
│   ├── models.py
│   └── recommender.py
├── app.py                                   ← Streamlit dashboard
├── requirements.txt
└── README.md
```

---
*Built by **Latha Iyer** | M.S. Business Analytics, University of Louisville*  
*Stack: Python · pandas · scikit-learn · XGBoost · ALS (Matrix Factorization) · matplotlib · seaborn · Streamlit*


---
---
# 🤖 Part 2 — Agentic AI Extensions (Modules 10–13)
*The following modules layer Claude API intelligence on top of the analysis above.*  
*`anthropic` and `statsmodels` are the only additional dependencies.*


---
## 🗣️ Module 10 — LLM Insight Narrator
*After each chart, a `ClaudeNarrator` agent reads the actual computed values and returns  
live business commentary — no hardcoded insight strings.*


In [ ]:
# ── ClaudeNarrator ────────────────────────────────────────────────────────────
class ClaudeNarrator:
    SYSTEM = (
        "You are a senior marketing analytics advisor. "
        "Given JSON data from a campaign analysis, write 2-3 sentences of "
        "sharp, actionable insight for a VP of Marketing. "
        "Be specific about numbers. Plain prose — no bullet points."
    )
    def __init__(self, model="claude-sonnet-4-20250514"):
        self.client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from Colab Secrets
        self.model  = model

    def narrate(self, context: str, data: dict) -> str:
        payload = json.dumps(data, indent=2)
        prompt  = f"Context: {context}\n\nData:\n{payload}\n\nInsight:"
        try:
            msg = self.client.messages.create(
                model=self.model, max_tokens=300,
                system=self.SYSTEM,
                messages=[{"role":"user","content":prompt}]
            )
            return msg.content[0].text.strip()
        except Exception as e:
            return f"[Narrator offline — set ANTHROPIC_API_KEY in Colab Secrets. {e}]"

narrator = ClaudeNarrator()
print("✅ ClaudeNarrator ready")
print("   Add ANTHROPIC_API_KEY to Colab Secrets (🔑 sidebar) for live narration.")


In [ ]:
# ── 10.1  Response rate by segment + live narration ──────────────────────────
fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Campaign Response Rate by Segment', fontsize=14, fontweight='bold')
seg_stats = {}
for ax, col, title in zip(axes,
    ['Age_Group','Income_Group','Education'],
    ['Age Group','Income Group','Education Level']):
    rr = df.groupby(col)['Response'].mean().sort_values(ascending=False)
    bars = ax.bar(rr.index, rr.values*100, color=PALETTE[:len(rr)], edgecolor='white', linewidth=1.5)
    for bar,v in zip(bars,rr.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{v:.1%}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontweight='bold'); ax.set_ylabel('Response Rate (%)')
    ax.set_ylim(0, rr.max()*1.35); ax.tick_params(axis='x', rotation=20)
    sns.despine(ax=ax); seg_stats[col] = {str(k): round(float(v),4) for k,v in rr.items()}
plt.tight_layout(); plt.show()

print("\n🤖 Claude Narrator:")
print("-"*60)
insight = narrator.narrate(
    "Campaign response rates by age, income and education segment",
    {"segment_response_rates": seg_stats, "overall_rate": round(float(y.mean()),4)}
)
print(textwrap.fill(insight, width=78))


In [ ]:
# ── 10.2  Feature importance + narration ─────────────────────────────────────
fi = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,5))
fi.head(10).sort_values().plot(kind='barh', ax=ax, color=PALETTE[0], edgecolor='white')
ax.set_title('XGBoost — Top 10 Feature Importances', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score'); sns.despine(ax=ax)
plt.tight_layout(); plt.show()

print("\n🤖 Claude Narrator:")
print("-"*60)
insight2 = narrator.narrate(
    "XGBoost feature importances for campaign response prediction",
    {"top_10_features": fi.head(10).round(4).to_dict(),
     "model_auc": round(auc_xgb,4)}
)
print(textwrap.fill(insight2, width=78))


In [ ]:
# ── 10.3  Model comparison narration ─────────────────────────────────────────
rf_p  = rf.predict(X_test);  xgb_p = xgb.predict(X_test)
metrics = {
    "Random Forest": {"AUC":round(auc_rf,4),"Precision":round(precision_score(y_test,rf_p),4),
                      "Recall":round(recall_score(y_test,rf_p),4),"F1":round(f1_score(y_test,rf_p),4)},
    "XGBoost":       {"AUC":round(auc_xgb,4),"Precision":round(precision_score(y_test,xgb_p),4),
                      "Recall":round(recall_score(y_test,xgb_p),4),"F1":round(f1_score(y_test,xgb_p),4)},
}
print("📊 Model Metrics:"); print(pd.DataFrame(metrics).T.to_string())

print("\n🤖 Claude Narrator:")
print("-"*60)
insight3 = narrator.narrate(
    "Random Forest vs XGBoost comparison for campaign response prediction",
    {"metrics": metrics, "test_size": len(y_test), "positive_rate": round(float(y_test.mean()),3)}
)
print(textwrap.fill(insight3, width=78))


---
## 🧪 Module 11 — A/B Testing Agent
*Designs, runs, and interprets campaign experiments with sequential monitoring.*

**Three-step pipeline:**  
`design()` → power analysis → `simulate_and_monitor()` → sequential checkpoints → `decide()` → Claude verdict


In [ ]:
# ── ABTestAgent ───────────────────────────────────────────────────────────────
class ABTestAgent:
    def __init__(self, narrator, alpha=0.05, power=0.80):
        self.narrator = narrator
        self.alpha    = alpha
        self.power    = power

    def design(self, baseline_rate, mde):
        treat_rate  = baseline_rate + mde
        pooled_std  = np.sqrt((baseline_rate*(1-baseline_rate)+treat_rate*(1-treat_rate))/2)
        effect_size = abs(treat_rate - baseline_rate) / pooled_std
        z_a = stats.norm.ppf(1 - self.alpha/2)
        z_b = stats.norm.ppf(self.power)
        n   = int(np.ceil(((z_a + z_b) / effect_size)**2))
        return {"baseline_rate":baseline_rate,"mde":mde,"treatment_rate":round(treat_rate,4),
                "effect_size":round(effect_size,4),"n_per_arm":n,"total_n":2*n}

    def simulate_and_monitor(self, design, true_lift=0.0, checkpoints=5):
        n   = design['n_per_arm']
        b   = design['baseline_rate']
        t   = b + true_lift
        # O'Brien-Fleming alpha spending
        a_spend = [self.alpha * (k/checkpoints)**0.5 for k in range(1,checkpoints+1)]
        ctrl_c  = np.random.binomial(1, b, n)
        treat_c = np.random.binomial(1, t, n)
        results = []
        step    = n // checkpoints
        for k, a in enumerate(a_spend, 1):
            nk  = min(step*k, n)
            c_k = ctrl_c[:nk].sum();  t_k = treat_c[:nk].sum()
            pc  = c_k/nk;             pt  = t_k/nk
            _, pval = proportions_ztest([t_k,c_k],[nk,nk],alternative="larger")
            results.append({"checkpoint":k,"n_per_arm":nk,
                            "control_rate":round(pc,4),"treatment_rate":round(pt,4),
                            "lift":round(pt-pc,4),"p_value":round(float(pval),4),
                            "alpha_threshold":round(a,4),"significant":bool(pval<a)})
        return results

    def decide(self, design, monitoring):
        return self.narrator.narrate(
            "A/B test for email campaign variant. Recommend: launch / extend / stop.",
            {"design":design,"final":monitoring[-1],
             "early_stops":sum(1 for r in monitoring if r['significant'])}
        )

ab_agent = ABTestAgent(narrator)
print("✅ ABTestAgent ready")


In [ ]:
# ── Run experiment ────────────────────────────────────────────────────────────
baseline   = float(y.mean())
design     = ab_agent.design(baseline, mde=0.04)
monitoring = ab_agent.simulate_and_monitor(design, true_lift=0.035, checkpoints=5)

print("📐 Experiment Design")
print(f"   Baseline rate : {design['baseline_rate']:.1%}")
print(f"   MDE           : {design['mde']:+.1%}")
print(f"   Required n/arm: {design['n_per_arm']:,}")
print(f"\n📈 Sequential Monitoring (O'Brien-Fleming)")
mon_df = pd.DataFrame(monitoring)
print(mon_df[['checkpoint','n_per_arm','control_rate','treatment_rate',
              'lift','p_value','alpha_threshold','significant']].to_string(index=False))

# ── Chart ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(mon_df.checkpoint, mon_df.control_rate*100,
             'o-', color=PALETTE[2], lw=2, label='Control')
axes[0].plot(mon_df.checkpoint, mon_df.treatment_rate*100,
             's-', color=PALETTE[0], lw=2.5, label='Treatment')
axes[0].fill_between(mon_df.checkpoint,
                     mon_df.control_rate*100, mon_df.treatment_rate*100,
                     alpha=0.12, color=PALETTE[0])
axes[0].set_title('Conversion Rate Over Checkpoints', fontweight='bold')
axes[0].set_xlabel('Checkpoint'); axes[0].set_ylabel('Conversion Rate (%)')
axes[0].legend(); sns.despine(ax=axes[0])

axes[1].bar(mon_df.checkpoint, mon_df.p_value,
            color=PALETTE[0], edgecolor='white', width=0.5)
axes[1].plot(mon_df.checkpoint, mon_df.alpha_threshold,
             'r--', lw=2, label="Alpha threshold (O'B-F)")
for _, row in mon_df.iterrows():
    if row.significant:
        axes[1].text(row.checkpoint, row.p_value+0.002, '✓',
                     ha='center', color='green', fontsize=14, fontweight='bold')
axes[1].set_title('p-value vs Alpha Spending', fontweight='bold')
axes[1].set_xlabel('Checkpoint'); axes[1].set_ylabel('p-value')
axes[1].legend(); sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig('ab_test_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🤖 ABTestAgent Decision:")
print("="*60)
print(textwrap.fill(ab_agent.decide(design, monitoring), width=78))


---
## 🎯 Module 12 — Causal Uplift Model (T-Learner)
*Goes beyond "who will respond?" → "who responds **because of** the campaign?"*

| Segment | Responds to Campaign | Would Buy Anyway |
|---------|---------------------|-----------------|
| 🟢 **Persuadable** | ✅ | ❌ → **Target these** |
| 🟡 **Sure Thing** | ✅ | ✅ → Safe to skip |
| 🔴 **Lost Cause** | ❌ | ❌ → Don't waste budget |
| ⚪ **Sleeping Dog** | ❌ | ✅ → Risk annoying them |


In [ ]:
# ── T-Learner Uplift Model ────────────────────────────────────────────────────
class TLearnerUplift:
    def __init__(self):
        kw = dict(n_estimators=150,max_depth=4,learning_rate=0.1,
                  random_state=42,verbosity=0,eval_metric='logloss')
        self.m1 = XGBClassifier(**kw)   # trained on treated group
        self.m0 = XGBClassifier(**kw)   # trained on control group

    def fit(self, X, y, T):
        self.m1.fit(X[T==1], y[T==1])
        self.m0.fit(X[T==0], y[T==0])
        return self

    def predict_cate(self, X):
        return self.m1.predict_proba(X)[:,1] - self.m0.predict_proba(X)[:,1]

    def segment(self, X, cate_thresh=0.05, p0_thresh=0.30):
        cate = self.predict_cate(X)
        p0   = self.m0.predict_proba(X)[:,1]
        segs = np.where((cate>=cate_thresh)&(p0<p0_thresh),  'Persuadable',
               np.where((cate>=cate_thresh)&(p0>=p0_thresh), 'Sure Thing',
               np.where((cate<cate_thresh)&(p0>=p0_thresh),  'Lost Cause',
                                                               'Sleeping Dog')))
        return segs, cate

# ── Simulate random treatment assignment ─────────────────────────────────────
T_assign = (np.random.rand(len(X)) < 0.5).astype(int)
uplift   = TLearnerUplift()
uplift.fit(X.values, y.values, T_assign)
segments, cate = uplift.segment(X.values)

seg_counts = pd.Series(segments).value_counts()
print("✅ T-Learner trained")
print("\n📊 Customer Segments:")
SEG_COLORS = {'Persuadable':'#0A9396','Sure Thing':'#EE9B00',
              'Lost Cause':'#AE2012','Sleeping Dog':'#94D2BD'}
for seg,cnt in seg_counts.items():
    pct = cnt/len(segments)*100
    bar = '█'*int(pct/2)
    print(f"  {seg:15s}: {cnt:5,}  ({pct:.1f}%)  {bar}")
print(f"\n  Mean CATE : {cate.mean():+.4f}")
print(f"  Persuadable pool → {(segments=='Persuadable').sum():,} customers (highest ROI targets)")


In [ ]:
# ── Segment visualisation ────────────────────────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(17,5))
fig.suptitle('Causal Uplift — T-Learner Results', fontsize=14, fontweight='bold')

counts = pd.Series(segments).value_counts()
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[SEG_COLORS[s] for s in counts.index],
            wedgeprops={'edgecolor':'white','linewidth':2}, startangle=90)
axes[0].set_title('Segment Distribution')

for seg,color in SEG_COLORS.items():
    mask = segments==seg
    if mask.sum()>10:
        axes[1].hist(cate[mask],bins=30,alpha=0.55,color=color,label=seg,edgecolor='white')
axes[1].axvline(0,color='black',lw=1.5,linestyle='--',alpha=0.6)
axes[1].set_title('CATE Distribution by Segment')
axes[1].set_xlabel('Estimated Treatment Effect'); axes[1].set_ylabel('Count')
axes[1].legend(fontsize=8); sns.despine(ax=axes[1])

df_seg = df.copy(); df_seg['Segment'] = segments; df_seg['CATE'] = cate
spend = df_seg.groupby('Segment')['Total_Spend'].mean().sort_values()
bars  = axes[2].barh(spend.index, spend.values,
                     color=[SEG_COLORS[s] for s in spend.index], edgecolor='white')
for bar,val in zip(bars,spend.values):
    axes[2].text(val+10, bar.get_y()+bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontweight='bold', fontsize=9)
axes[2].set_title('Avg Total Spend by Segment'); axes[2].set_xlabel('Avg Spend ($)')
sns.despine(ax=axes[2])
plt.tight_layout()
plt.savefig('uplift_segments.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Qini uplift curve ────────────────────────────────────────────────────────
ev = pd.DataFrame({'cate':cate,'y':y.values,'T':T_assign})
ev = ev.sort_values('cate',ascending=False).reset_index(drop=True)
ev['cum_t']  = (ev['T']==1).cumsum()
ev['cum_c']  = (ev['T']==0).cumsum()
ev['conv_t'] = ((ev['T']==1)&(ev['y']==1)).cumsum()
ev['conv_c'] = ((ev['T']==0)&(ev['y']==1)).cumsum()
ev['uplift'] = ev.conv_t/(ev.cum_t+1) - ev.conv_c/(ev.cum_c+1)
pct = np.linspace(0,100,len(ev))

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(pct, ev.uplift*100, color=PALETTE[0], lw=2.5)
axes[0].axhline(0,color='gray',lw=1,linestyle='--')
axes[0].fill_between(pct, ev.uplift*100, 0,
                     where=ev.uplift>0, alpha=0.15, color=PALETTE[0])
axes[0].set_title('Qini Uplift Curve', fontweight='bold')
axes[0].set_xlabel('% Population Targeted (ranked by CATE)')
axes[0].set_ylabel('Incremental Conversion Lift (%)')
sns.despine(ax=axes[0])

df_seg['Income_Group'] = pd.cut(df_seg['Income'],
    bins=[0,30000,60000,90000,200000], labels=['Low','Mid','High','Premium'])
cate_inc = df_seg.groupby('Income_Group')['CATE'].mean()
bars = axes[1].bar(cate_inc.index, cate_inc.values*100,
                   color=PALETTE[:4], edgecolor='white', width=0.55)
for bar,val in zip(bars,cate_inc.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.03,
                 f'{val:+.2f}pp', ha='center', fontweight='bold', fontsize=10)
axes[1].axhline(0,color='black',lw=0.8,linestyle='--',alpha=0.4)
axes[1].set_title('Mean CATE by Income Group', fontweight='bold')
axes[1].set_ylabel('Mean CATE (pp)'); sns.despine(ax=axes[1])
plt.tight_layout()
plt.savefig('uplift_qini.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n🤖 Claude Narrator — Uplift Insight:")
print("-"*60)
insight_up = narrator.narrate(
    "T-Learner causal uplift model results for campaign targeting",
    {"segments": {s:int(v) for s,v in seg_counts.items()},
     "mean_cate": round(float(cate.mean()),4),
     "persuadable_pct": round(float((segments=='Persuadable').mean()*100),1),
     "cate_by_income": {str(k):round(float(v),4) for k,v in cate_inc.items()}}
)
print(textwrap.fill(insight_up, width=78))


---
## 🏭 Module 13 — Multi-Agent Pipeline
*Three specialised agents that hand off structured context to each other.*

```
┌─────────────┐     ┌──────────────┐     ┌─────────────┐
│  DataAgent  │────►│  ModelAgent  │────►│ ReportAgent │
│             │     │              │     │             │
│ • Profile   │     │ • Evaluate   │     │ • Synthesise│
│ • QA flags  │     │ • Flag risks │     │ • Exec summ │
│ • Top feats │     │ • Recommend  │     │ • Next steps│
└─────────────┘     └──────────────┘     └─────────────┘
```


In [ ]:
# ── Base agent class ─────────────────────────────────────────────────────────
class BaseAgent:
    def __init__(self, name, system_prompt, narrator):
        self.name     = name
        self.system   = system_prompt
        self.narrator = narrator
        self.memory   = []

    def think(self, task, data):
        try:
            msg = self.narrator.client.messages.create(
                model=self.narrator.model, max_tokens=512,
                system=self.system,
                messages=[{"role":"user",
                           "content":f"Task: {task}\n\nData:\n{json.dumps(data,indent=2)}"}]
            )
            return msg.content[0].text.strip()
        except Exception as e:
            return f"[{self.name} offline — {e}]"

print("✅ BaseAgent defined")


In [ ]:
# ── DataAgent ─────────────────────────────────────────────────────────────────
class DataAgent(BaseAgent):
    SYSTEM = ("You are a senior data quality engineer. Profile a marketing dataset, "
              "flag anomalies, report class imbalance, and recommend top features "
              "for the modelling team. Be concise and technical. 3-4 sentences.")

    def __init__(self, narrator):
        super().__init__("DataAgent", self.SYSTEM, narrator)

    def run(self, df, features, target):
        print(f"[{self.name}] Profiling dataset...")
        profile = {
            "n_rows": len(df), "n_features": len(features),
            "positive_rate": round(float(df[target].mean()),4),
            "missing_pct":   round(float(df[features].isnull().mean().mean()*100),2),
            "class_balance": {str(k):int(v) for k,v in df[target].value_counts().items()},
            "top_correlations": {
                k: round(float(v),4) for k,v in
                df[features+[target]].corr()[target].drop(target)
                .abs().sort_values(ascending=False).head(5).items()
            },
        }
        commentary = self.think(
            "Profile this dataset. Flag quality issues, note class imbalance, "
            "and name the 3 most predictive features.", profile)
        output = {"profile": profile, "commentary": commentary}
        self.memory.append(output)
        print(f"  ✓ {len(df):,} rows  |  {profile['positive_rate']:.1%} positive rate")
        print(f"\n  💬 DataAgent: {textwrap.fill(commentary,75)}")
        return output

# ── ModelAgent ────────────────────────────────────────────────────────────────
class ModelAgent(BaseAgent):
    SYSTEM = ("You are an ML engineer reviewing model performance for a marketing system. "
              "Diagnose issues, compare models, flag risks, and suggest one specific "
              "improvement experiment. Precise numbers. 3-4 sentences.")

    def __init__(self, narrator):
        super().__init__("ModelAgent", self.SYSTEM, narrator)

    def run(self, model_results, data_output):
        print(f"\n[{self.name}] Evaluating models...")
        issues = []
        rf_r   = model_results['rf']['recall']
        xgb_r  = model_results['xgboost']['recall']
        if min(rf_r, xgb_r) < 0.30:
            issues.append("Low recall on positive class — consider threshold tuning")
        if abs(model_results['rf']['auc']-model_results['xgboost']['auc']) < 0.01:
            issues.append("AUC gap <0.01 — ensemble may not add marginal value")
        if data_output['profile']['positive_rate'] < 0.25:
            issues.append("Class imbalance — SMOTE or class_weight adjustment recommended")
        payload   = {**model_results, "issues": issues,
                     "data_context": data_output['profile']}
        commentary = self.think(
            "Evaluate these models, flag issues, recommend deployment candidate "
            "and one improvement experiment.", payload)
        output = {"results": model_results, "issues": issues, "commentary": commentary}
        self.memory.append(output)
        print(f"  ✓ {len(issues)} issue(s) flagged")
        for iss in issues:
            print(f"    ⚠️  {iss}")
        print(f"\n  💬 ModelAgent: {textwrap.fill(commentary,75)}")
        return output

# ── ReportAgent ───────────────────────────────────────────────────────────────
class ReportAgent(BaseAgent):
    SYSTEM = ("You are a senior analytics consultant writing an executive summary for a CMO. "
              "Synthesise data quality, model performance, and causal uplift results "
              "into clear business language. End with exactly 3 numbered next steps. "
              "Max 6 sentences total.")

    def __init__(self, narrator):
        super().__init__("ReportAgent", self.SYSTEM, narrator)

    def run(self, data_out, model_out, uplift_summary):
        print(f"\n[{self.name}] Writing executive summary...")
        payload  = {"data": data_out['profile'], "models": model_out['results'],
                    "model_issues": model_out['issues'], "uplift": uplift_summary}
        report   = self.think("Write CMO executive summary with 3 numbered next steps.", payload)
        self.memory.append({"report": report})
        return report

print("✅ DataAgent, ModelAgent, ReportAgent defined")


In [ ]:
# ── Run the full pipeline ─────────────────────────────────────────────────────
print("="*62)
print("  🏭  MULTI-AGENT PIPELINE — STARTING")
print("="*62)

# Agent 1 — Data
data_agent  = DataAgent(narrator)
data_out    = data_agent.run(df, FEATURES, 'Response')

# Agent 2 — Model
rf_p = rf.predict(X_test); xgb_p = xgb.predict(X_test)
model_results = {
    "rf":      {"auc":round(auc_rf,4), "precision":round(precision_score(y_test,rf_p),4),
                "recall":round(recall_score(y_test,rf_p),4), "f1":round(f1_score(y_test,rf_p),4)},
    "xgboost": {"auc":round(auc_xgb,4), "precision":round(precision_score(y_test,xgb_p),4),
                "recall":round(recall_score(y_test,xgb_p),4), "f1":round(f1_score(y_test,xgb_p),4)},
}
model_agent = ModelAgent(narrator)
model_out   = model_agent.run(model_results, data_out)

# Agent 3 — Report
uplift_summary = {
    "persuadable_count": int((segments=='Persuadable').sum()),
    "persuadable_pct":   round(float((segments=='Persuadable').mean()*100),1),
    "mean_cate":         round(float(cate.mean()),4),
    "incremental_convs": int((segments=='Persuadable').sum()*max(cate[segments=='Persuadable'].mean(),0)),
    "recommended_target": "Persuadable segment only",
}
report_agent = ReportAgent(narrator)
exec_summary = report_agent.run(data_out, model_out, uplift_summary)

print("\n" + "="*62)
print("  📋  EXECUTIVE SUMMARY  (ReportAgent via Claude)")
print("="*62)
print(textwrap.fill(exec_summary, width=78))
print("="*62)


In [ ]:
# ── Pipeline audit trail + visualisation ─────────────────────────────────────
print("\n📜 Agent Audit Trail")
for ag in [data_agent, model_agent, report_agent]:
    print(f"  {ag.name:15s}: {len(ag.memory)} output(s) stored in memory")

# Pipeline diagram
fig, ax = plt.subplots(figsize=(13,3.5))
ax.set_xlim(0,13); ax.set_ylim(0,3.5); ax.axis('off')
fig.patch.set_facecolor('#F8F9FA')
boxes = [
    (1.2, "DataAgent",   "Profiles data\nFlags QA issues\nTop features"),
    (5.2, "ModelAgent",  "Evaluates RF/XGB\nFlags low recall\nRecommends model"),
    (9.2, "ReportAgent", "Synthesises all\nExecutive summary\n3 next steps"),
]
for x, title, desc in boxes:
    rect = plt.Rectangle((x,0.8),3,1.8, facecolor=PALETTE[0], edgecolor='white',
                          linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x+1.5,2.1, title, ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')
    ax.text(x+1.5,1.4, desc,  ha='center', va='center',
            fontsize=8.5, color='white', alpha=0.9)
for x in [4.2, 8.2]:
    ax.annotate('', xy=(x,1.7), xytext=(x-0.05,1.7),
                arrowprops=dict(arrowstyle='->', color=PALETTE[1], lw=2.5))
ax.text(6.5,3.15, 'Multi-Agent Marketing Analytics Pipeline',
        ha='center', fontsize=13, fontweight='bold', color=PALETTE[2])
plt.tight_layout()
plt.savefig('pipeline_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ All 4 agentic modules complete!")


---
## ✅ Summary

| Module | Status | Key Output |
|--------|--------|-----------|
| 10 — LLM Narrator | ✅ | Live Claude commentary on every chart |
| 11 — A/B Testing Agent | ✅ | Sequential monitoring + causal launch decision |
| 12 — Causal Uplift (T-Learner) | ✅ | Persuadable segment + Qini curve |
| 13 — Multi-Agent Pipeline | ✅ | DataAgent → ModelAgent → ReportAgent + exec summary |

---
**Next:** Push both notebooks to GitHub and link from resume.  
`github.com/lathaiyer/marketing-analytics-platform`

*Latha Iyer · M.S. Business Analytics, University of Louisville*
